# Figure S1 — Raw frequency heatmap

Displays a heatmap of log-transformed raw Twitter term frequencies
across 28 languages over all weeks in the dataset.

**Inputs:** `CHOSEN_WEEKLY_PIVOT_FILE` (`data/processed/chosen_words_weekly_pivoted.csv`)
**Outputs:** `outputs/figures/Fig.S1_raw_freq_heatmap/`
**Prerequisites:** run `02_combine_data.ipynb` first

In [ ]:
import sys
sys.path.insert(0, '..')
from config import CHOSEN_WEEKLY_PIVOT_FILE, WORD_FORMS_ALL, FIGURES_DIR

import pandas as pd
import numpy as np
from datetime import datetime
import plotly.graph_objects as go
import plotly.io as pio

pio.templates.default = "plotly_white"

## ⚙️ Parameters

In [ ]:
# Set to True to exclude Ukrainian and Russian from the heatmap
remove_UA_RU = False

# Row order for the heatmap (top → bottom)
LANGUAGE_ORDER = [
    'Ukrainian', 'Russian', 'Arabic', 'Portuguese', 'Catalan', 'Korean',
    'Persian', 'Turkish', 'Indonesian', 'Urdu', 'Vietnamese',
    'Serbian', 'Estonian', 'Romanian', 'Greek', 'Hungarian', 'Polish',
    'Swedish', 'Czech', 'Spanish', 'Danish', 'English', 'Dutch',
    'Norwegian', 'Finnish', 'French', 'Italian', 'German'
]

## Load & prepare data

In [ ]:
DFfreq = pd.read_csv(CHOSEN_WEEKLY_PIVOT_FILE, index_col=0)

# Robust index handling: supports both ISO-index and language-index pivots
meta_lang = pd.read_csv(WORD_FORMS_ALL, usecols=['ISO', 'Language']).drop_duplicates()
iso_to_name = meta_lang.set_index('ISO')['Language'].to_dict()
iso_set = set(meta_lang['ISO'])
name_set = set(meta_lang['Language'])

idx = pd.Index(DFfreq.index.astype(str))
if idx.isin(iso_set).all():
    DFfreq.index = idx.map(iso_to_name)
elif idx.isin(name_set).all():
    DFfreq.index = idx
else:
    DFfreq.index = idx.map(iso_to_name).fillna(idx)
DFfreq.index = pd.Index(DFfreq.index, name='language')

if remove_UA_RU:
    DFfreq = DFfreq.drop(index=['Ukrainian', 'Russian'], errors='ignore')

# Reorder rows to LANGUAGE_ORDER (only rows present in data)
order = [lang for lang in LANGUAGE_ORDER if lang in DFfreq.index]
DFfreq = DFfreq.reindex(order)

# Log-transform: zeros → NaN (plotted as white), positives → log10(x * 100000)
DFfreqLog = DFfreq.map(lambda x: np.nan if x == 0 else np.log10(x * 100_000))

# Non-zero raw values used to build informative colorbar tick labels
nonzero_vals = DFfreq.values.flatten()
nonzero_vals = nonzero_vals[nonzero_vals > 0]
min_percent = nonzero_vals.min() * 100
max_percent = nonzero_vals.max() * 100

DFfreqLog.head(3)

## Heatmap plot

In [ ]:
min_val = float(np.nanmin(DFfreqLog.values))
max_val = float(np.nanmax(DFfreqLog.values))

custom_colorscale = [
    [0, "#f0f0f0"],  # very light gray (min non-zero)
    [1, "#000000"],  # black (max)
]

# Colorbar ticks expressed as % of total Twitter corpus
def percent_to_log(val):
    if val == 0:
        return -1.5  # placeholder below real data range
    return np.log10(val / 100 * 100_000)

linear_tick_percentages = sorted(set([0.0001, 0.001, 0.01, 0.1, 1, min_percent, max_percent]))
tickvals = [percent_to_log(v) for v in linear_tick_percentages]
ticktext = [f"{v:.4g}%" for v in linear_tick_percentages]

fig = go.Figure()

fig.add_trace(go.Heatmap(
    z=DFfreqLog.values,
    x=DFfreqLog.columns,
    y=DFfreqLog.index,
    colorscale=custom_colorscale,
    zmin=min_val,
    zmax=max_val,
    colorbar=dict(
        orientation="h",
        y=-0.25,
        x=0.5,
        xanchor="center",
        len=0.50,
        thickness=45,
        thicknessmode="pixels",
        outlinewidth=2,
        outlinecolor="black",
        ticks="outside",
        tickvals=tickvals,
        ticktext=ticktext,
        ticklabelposition="outside bottom",
        ticklen=10,
        tickcolor="black",
        tickfont=dict(size=29, color="black"),
    ),
))

fig.update_layout(
    height=1400,
    width=2100,
    margin=dict(l=70, r=40, t=60, b=200),
    xaxis=dict(title=" ", tickfont=dict(size=34, color="black")),
    yaxis=dict(title=" ", tickfont=dict(size=31, color="black"), autorange="reversed"),
    title=dict(text="", font=dict(size=30, color="black")),
)

fig.show()

## Save figure

In [ ]:
fig_dir = FIGURES_DIR / "Fig.S1_raw_freq_heatmap"
plotname = "rawfreqplot"
suffix = "_NO-UA-RU" if remove_UA_RU else "_with-UA-RU"
name = f"{plotname}{suffix}"

fig_dir.mkdir(parents=True, exist_ok=True)

for fmt in ['pdf', 'svg', 'jpeg']:
    out = fig_dir / fmt
    out.mkdir(exist_ok=True)
    fig.write_image(str(out / f"{name}.{fmt}"), format=fmt, engine="kaleido")
    print(f"Saved {fmt.upper()}: {out / f'{name}.{fmt}'}")

html_path = fig_dir / f"{name}.html"
fig.write_html(str(html_path))
print(f"Saved HTML: {html_path}")